# **TEMA: CODIFICACIÓN DE VARIABLES CATEGÓRICAS**
## NOMBRE: CARLOS ENRIQUE NIEVES OCHOA.


In [1]:
# Archivo de ejercicio de aguas

import pandas as pd
import numpy as np
import statsmodels.api as sm

df = pd.read_excel("EJERCICIO - AGUAS2.xlsx")

df

,Sabor,Tamaño,Hielo,Calificación
0,Naranja,Chica,Sí,8.5
1,Naranja,Chica,No,7.0
2,Naranja,Grande,Sí,9.0
3,Naranja,Grande,No,7.5
4,Coco,Chica,Sí,7.5
5,Coco,Chica,No,6.0
6,Coco,Grande,Sí,8.0
7,Coco,Grande,No,7.0
8,Kahlúa,Chica,Sí,9.0
9,Kahlúa,Chica,No,8.0


In [2]:
df.head()

,Sabor,Tamaño,Hielo,Calificación
0,Naranja,Chica,Sí,8.5
1,Naranja,Chica,No,7.0
2,Naranja,Grande,Sí,9.0
3,Naranja,Grande,No,7.5
4,Coco,Chica,Sí,7.5


In [3]:
df.tail()

,Sabor,Tamaño,Hielo,Calificación
15,Fresa-piña,Grande,No,6.0
16,Sandía,Chica,Sí,7.0
17,Sandía,Chica,No,5.0
18,Sandía,Grande,Sí,7.5
19,Sandía,Grande,No,6.0


In [4]:
df.describe()

,Calificación
count,20.000000
mean,7.600000
std,1.262829
min,5.000000
25%,7.000000
50%,7.500000
75%,8.500000
max,10.000000


## Codificar Tamaño como Chica = 0 y Grande = 1, e Hielo como No = 0 y Sí = 1


In [5]:
# Codificación utilizada en Actividad4
df["Tamaño"] = df["Tamaño"].map({"Chica": 0, "Grande": 1})
df["Hielo"] = df["Hielo"].map({"No": 0, "Sí": 1})

df


,Sabor,Tamaño,Hielo,Calificación
0,Naranja,0,1,8.5
1,Naranja,0,0,7.0
2,Naranja,1,1,9.0
3,Naranja,1,0,7.5
4,Coco,0,1,7.5
5,Coco,0,0,6.0
6,Coco,1,1,8.0
7,Coco,1,0,7.0
8,Kahlúa,0,1,9.0
9,Kahlúa,0,0,8.0


## Modificación de tabla

### Establecer 0 y 1 dependiendo del sabor.

In [6]:
# Crear una columna por cada sabor
for sabor in df["Sabor"].unique():
    df[sabor] = np.where(
        df["Sabor"].str.contains(sabor, case=False, na=False, regex=False),
        1,
        0
    )

# Quitar Sabor y conservar las demás columnas
df = df.drop(columns=["Sabor"])

df


,Tamaño,Hielo,Calificación,Naranja,Coco,Kahlúa,Fresa-piña,Sandía
0,0,1,8.5,1,0,0,0,0
1,0,0,7.0,1,0,0,0,0
2,1,1,9.0,1,0,0,0,0
3,1,0,7.5,1,0,0,0,0
4,0,1,7.5,0,1,0,0,0
5,0,0,6.0,0,1,0,0,0
6,1,1,8.0,0,1,0,0,0
7,1,0,7.0,0,1,0,0,0
8,0,1,9.0,0,0,1,0,0
9,0,0,8.0,0,0,1,0,0


## Con librería de OLS hacer una regresión múltiple como variables explicativas de columnas de 1, sabor, tamaño, hielo y con y como la calificación.

In [7]:
# Calificación como variable dependiente
y = df["Calificación"]

# Conservar los cinco sabores, tamaño y hielo, como en Actividad4
columnas = ["Naranja", "Coco", "Kahlúa", "Fresa-piña", "Sandía", "Tamaño", "Hielo"]
X = df[columnas].astype(float)
X.insert(0, "Unos", 1.0)

# Estimar OLS mediante la pseudoinversa
modelo = sm.OLS(y, X).fit(method="pinv")
print(modelo.summary())


                            OLS Regression Results                            
Dep. Variable:           Calificación   R-squared:                       0.927
Model:                            OLS   Adj. R-squared:                  0.894
Method:                 Least Squares   F-statistic:                     27.67
Date:                Thu, 03 Sep 2026   Prob (F-statistic):           1.11e-06
Time:                        17:46:10   Log-Likelihood:                -6.3060
No. Observations:                  20   AIC:                             26.61
Df Residuals:                      13   BIC:                             33.58
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Unos           5.4583      0.133     41.111      0.0

C:\Users\carlo\AppData\Local\Temp\ipykernel_16712\1297358075.py:10: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  modelo = sm.OLS(y, X).fit(method="pinv")


### Modelo estimado

Calificación = 5.4583 + 1.4917(Naranja) + 0.6167(Coco) + 2.3667(Kahlúa) + 1.1167(Fresa-piña) - 0.1333(Sandía) + 0.5000(Tamaño) + 1.6000(Hielo).

## Realizar el mismo procedimiento pero con librerías de Sklearn.preprocessing, OrdinalEncoder y OneHotEnconder.

In [8]:
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder

# Recuperar las categorías originales

df_sklearn = pd.read_excel("EJERCICIO - AGUAS2.xlsx")

# Mantener Chica = 0, Grande = 1, No = 0 y Sí = 1

ordinal = OrdinalEncoder(
    categories=[["Chica", "Grande"], ["No", "Sí"]],
    dtype=int
)
df_sklearn[["Tamaño", "Hielo"]] = ordinal.fit_transform(
    df_sklearn[["Tamaño", "Hielo"]]
)

df_sklearn

,Sabor,Tamaño,Hielo,Calificación
0,Naranja,0,1,8.5
1,Naranja,0,0,7.0
2,Naranja,1,1,9.0
3,Naranja,1,0,7.5
4,Coco,0,1,7.5
5,Coco,0,0,6.0
6,Coco,1,1,8.0
7,Coco,1,0,7.0
8,Kahlúa,0,1,9.0
9,Kahlúa,0,0,8.0


In [9]:
# Crear una columna indicadora por cada sabor

onehot = OneHotEncoder(
    categories=[["Naranja", "Coco", "Kahlúa", "Fresa-piña", "Sandía"]],
    sparse_output=False,
    dtype=int
)
sabores = onehot.fit_transform(df_sklearn[["Sabor"]])

# Convertir el resultado en una tabla con los nombres de los sabores

df_sabores = pd.DataFrame(
    sabores,
    columns=onehot.categories_[0],
    index=df_sklearn.index
)

# Sustituir Sabor por sus columnas de 0 y 1

df_sklearn = pd.concat(
    [df_sklearn.drop(columns=["Sabor"]), df_sabores],
    axis=1
)

df_sklearn

,Tamaño,Hielo,Calificación,Naranja,Coco,Kahlúa,Fresa-piña,Sandía
0,0,1,8.5,1,0,0,0,0
1,0,0,7.0,1,0,0,0,0
2,1,1,9.0,1,0,0,0,0
3,1,0,7.5,1,0,0,0,0
4,0,1,7.5,0,1,0,0,0
5,0,0,6.0,0,1,0,0,0
6,1,1,8.0,0,1,0,0,0
7,1,0,7.0,0,1,0,0,0
8,0,1,9.0,0,0,1,0,0
9,0,0,8.0,0,0,1,0,0


In [10]:
# Repetir OLS con los datos codificados por sklearn
y_sklearn = df_sklearn["Calificación"]

# Mantener las mismas variables y el mismo orden que en el primer modelo
X_sklearn = df_sklearn[columnas].astype(float)
X_sklearn.insert(0, "Unos", 1.0)

modelo_sklearn = sm.OLS(y_sklearn, X_sklearn).fit(method="pinv")
print(modelo_sklearn.summary())


                            OLS Regression Results                            
Dep. Variable:           Calificación   R-squared:                       0.927
Model:                            OLS   Adj. R-squared:                  0.894
Method:                 Least Squares   F-statistic:                     27.67
Date:                Thu, 03 Sep 2026   Prob (F-statistic):           1.11e-06
Time:                        17:46:11   Log-Likelihood:                -6.3060
No. Observations:                  20   AIC:                             26.61
Df Residuals:                      13   BIC:                             33.58
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Unos           5.4583      0.133     41.111      0.0

C:\Users\carlo\AppData\Local\Temp\ipykernel_16712\4004399373.py:8: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  modelo_sklearn = sm.OLS(y_sklearn, X_sklearn).fit(method="pinv")


In [11]:
# Comparar los coeficientes de ambos procedimientos
comparacion = pd.DataFrame({"Pandas y NumPy": modelo.params, "Sklearn": modelo_sklearn.params})
comparacion.round(4)

,Pandas y NumPy,Sklearn
Unos,5.4583,5.4583
Naranja,1.4917,1.4917
Coco,0.6167,0.6167
Kahlúa,2.3667,2.3667
Fresa-piña,1.1167,1.1167
Sandía,-0.1333,-0.1333
Tamaño,0.5000,0.5000
Hielo,1.6000,1.6000
